# Inégalités de revenus et croissance économique dans les pays de l'OCDE (1990–2022)

**Auteur** : Mohammed Zahi — M1 MASERATI, Université Paris-Est Créteil (UPEC)  
**Contact** : z.mohammed91080@gmail.com · [LinkedIn](https://www.linkedin.com/in/mohammed-zahi-04a764343) · [Malt](https://www.malt.fr/profile/mohammedzahi)

---

## Question de recherche

> **Les inégalités de revenus freinent-elles la croissance économique dans les pays de l'OCDE ?**

Ce projet s'inscrit dans la littérature économique fondée sur les travaux d'**Ostry, Berg & Tsangarides (2014, FMI)** et d'**Aghion et al. (1999)**, qui montrent que les inégalités élevées peuvent réduire la croissance de long terme via des canaux comme la sous-investissement en capital humain, l'instabilité politique ou la compression de la demande intérieure.

## Compétences démontrées

- Collecte automatisée via **API Banque Mondiale** (World Development Indicators)
- Construction d'une **base de données panel** (pays × années)
- **Statistiques descriptives** et visualisations publication-ready
- **Régression OLS** avec variables de contrôle
- **Effets fixes** pays et temps (Within estimator)
- **Différence de différences** (choc de politiques redistributives)
- Tests de robustesse

---

## 1. Installation et imports

In [ ]:
# Installation des packages (décommenter si nécessaire)
# !pip install pandas numpy matplotlib seaborn statsmodels requests linearmodels

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
import warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, PooledOLS, BetweenOLS
from scipy import stats

warnings.filterwarnings('ignore')

# Style global
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
BLUE   = '#1a56db'
RED    = '#dc2626'
GREEN  = '#16a34a'
ORANGE = '#d97706'
GRAY   = '#6b7280'

print('Environnement prêt.')

## 2. Collecte des données via l'API Banque Mondiale

Tous les indicateurs sont téléchargés automatiquement — aucun fichier externe n'est nécessaire.

| Indicateur | Code WDI | Rôle dans le modèle |
|---|---|---|
| Croissance du PIB/hab. (%) | NY.GDP.PCAP.KD.ZG | **Variable dépendante** |
| Indice de Gini | SI.POV.GINI | **Variable d'intérêt** (inégalités) |
| PIB/hab. initial ($ 2015) | NY.GDP.PCAP.KD | Contrôle — convergence |
| Taux d'investissement (% PIB) | NE.GDI.TOTL.ZS | Contrôle — accumulation capital |
| Taux d'ouverture commerciale (% PIB) | NE.TRD.GNFS.ZS | Contrôle — intégration mondiale |
| Dépenses publiques éducation (% PIB) | SE.XPD.TOTL.GD.ZS | Contrôle — capital humain |
| Population (log) | SP.POP.TOTL | Contrôle — taille économie |

In [ ]:
# Pays OCDE (37 membres)
OCDE = [
    'AUS','AUT','BEL','CAN','CHL','COL','CZE','DNK','EST','FIN',
    'FRA','DEU','GRC','HUN','ISL','IRL','ISR','ITA','JPN','KOR',
    'LVA','LTU','LUX','MEX','NLD','NZL','NOR','POL','PRT','SVK',
    'SVN','ESP','SWE','CHE','TUR','GBR','USA'
]

NOMS_PAYS = {
    'AUS':'Australie','AUT':'Autriche','BEL':'Belgique','CAN':'Canada',
    'CHL':'Chili','COL':'Colombie','CZE':'Tchéquie','DNK':'Danemark',
    'EST':'Estonie','FIN':'Finlande','FRA':'France','DEU':'Allemagne',
    'GRC':'Grèce','HUN':'Hongrie','ISL':'Islande','IRL':'Irlande',
    'ISR':'Israël','ITA':'Italie','JPN':'Japon','KOR':'Corée du Sud',
    'LVA':'Lettonie','LTU':'Lituanie','LUX':'Luxembourg','MEX':'Mexique',
    'NLD':'Pays-Bas','NZL':'Nouvelle-Zélande','NOR':'Norvège','POL':'Pologne',
    'PRT':'Portugal','SVK':'Slovaquie','SVN':'Slovénie','ESP':'Espagne',
    'SWE':'Suède','CHE':'Suisse','TUR':'Turquie','GBR':'Royaume-Uni','USA':'États-Unis'
}

INDICATEURS = {
    'NY.GDP.PCAP.KD.ZG' : 'croissance_pib',
    'SI.POV.GINI'        : 'gini',
    'NY.GDP.PCAP.KD'     : 'pib_hab',
    'NE.GDI.TOTL.ZS'     : 'investissement',
    'NE.TRD.GNFS.ZS'     : 'ouverture',
    'SE.XPD.TOTL.GD.ZS'  : 'education',
    'SP.POP.TOTL'         : 'population',
}

def fetch_wb(indicator, countries, start=1990, end=2022):
    """Télécharge un indicateur WDI pour une liste de pays."""
    iso = ';'.join(countries)
    url = f'https://api.worldbank.org/v2/country/{iso}/indicator/{indicator}'
    params = {'format': 'json', 'per_page': 5000, 'date': f'{start}:{end}'}
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()[1]
        records = [
            {'iso3': e['countryiso3code'], 'annee': int(e['date']), 'valeur': float(e['value'])}
            for e in data if e['value'] is not None and e['countryiso3code'] in countries
        ]
        return pd.DataFrame(records)
    except Exception as ex:
        print(f'Erreur {indicator}: {ex}')
        return pd.DataFrame(columns=['iso3','annee','valeur'])

print('Téléchargement des données en cours...')
frames = {}
for code, nom in INDICATEURS.items():
    df_tmp = fetch_wb(code, OCDE)
    df_tmp = df_tmp.rename(columns={'valeur': nom})
    frames[nom] = df_tmp
    print(f'  {nom:25s} : {len(df_tmp):4d} observations')

print('\nTéléchargement terminé.')

## 3. Construction de la base panel

In [ ]:
# Fusion de tous les indicateurs
df = frames['croissance_pib'].copy()
for nom, df_ind in frames.items():
    if nom != 'croissance_pib':
        df = df.merge(df_ind, on=['iso3','annee'], how='outer')

# Métadonnées
df['pays'] = df['iso3'].map(NOMS_PAYS)
df['log_pib']        = np.log(df['pib_hab'].clip(lower=1))
df['log_population'] = np.log(df['population'].clip(lower=1))
df['decennie']       = (df['annee'] // 10) * 10

# Groupe inégalités : élevé si Gini > médiane, faible sinon
gini_median = df['gini'].median()
df['ineg_elevee'] = (df['gini'] > gini_median).astype(int)

# Choc politique redistributive — proxy : réforme fiscale post-2008
# On considère comme traité les pays dont le Gini a baissé de > 2 points entre 2007 et 2012
gini_avant = df[df['annee'] == 2007].set_index('iso3')['gini']
gini_apres = df[df['annee'] == 2012].set_index('iso3')['gini']
pays_traites = set((gini_avant - gini_apres).dropna()[lambda x: x > 2].index)
df['traite']    = df['iso3'].isin(pays_traites).astype(int)
df['post_2008'] = (df['annee'] >= 2008).astype(int)
df['did']       = df['traite'] * df['post_2008']

# Base nettoyée
VARS_MODELE = ['croissance_pib','gini','log_pib','investissement','ouverture','education','log_population']
df_clean = df.dropna(subset=VARS_MODELE).copy()
df_clean = df_clean.sort_values(['iso3','annee'])

print(f'Base panel finale :')
print(f'  Observations : {len(df_clean):,}')
print(f'  Pays         : {df_clean["iso3"].nunique()}')
print(f'  Période      : {df_clean["annee"].min()} – {df_clean["annee"].max()}')
print(f'  Pays traités (baisse Gini post-2008) : {len(pays_traites)}')
df_clean.head()

## 4. Statistiques descriptives

In [ ]:
print('=== Statistiques descriptives ===')
desc = df_clean[['croissance_pib','gini','investissement','ouverture','education']].describe().round(2)
print(desc)

print('\n=== Par groupe d\'inégalités ===')
print(df_clean.groupby('ineg_elevee')[['croissance_pib','gini']].agg(['mean','std']).round(2))

## 5. Visualisations

In [ ]:
# ─── Figure 1 : Distribution du Gini par décennie ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df_box = df_clean.dropna(subset=['gini','decennie'])
decennies = sorted(df_box['decennie'].unique())
data_box  = [df_box[df_box['decennie'] == d]['gini'].values for d in decennies]
bp = axes[0].boxplot(data_box, labels=[str(d)+'s' for d in decennies],
                     patch_artist=True, medianprops=dict(color='white', linewidth=2))
colors_box = [BLUE, GREEN, ORANGE, RED]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Distribution du Gini par décennie')
axes[0].set_ylabel('Indice de Gini')
axes[0].set_xlabel('Décennie')

# Scatter Gini vs Croissance
df_sc = df_clean.dropna(subset=['gini','croissance_pib'])
scatter_colors = [BLUE if g <= gini_median else RED for g in df_sc['gini']]
axes[1].scatter(df_sc['gini'], df_sc['croissance_pib'],
                c=scatter_colors, alpha=0.4, s=20, edgecolors='none')
# Droite de régression
z = np.polyfit(df_sc['gini'], df_sc['croissance_pib'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_sc['gini'].min(), df_sc['gini'].max(), 100)
axes[1].plot(x_line, p(x_line), color='black', linewidth=2, label=f'Tendance (pente={z[0]:.2f})')
axes[1].set_title('Gini vs Croissance du PIB/hab.')
axes[1].set_xlabel('Indice de Gini')
axes[1].set_ylabel('Croissance annuelle (%)')
axes[1].axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
axes[1].legend(fontsize=9)
axes[1].text(0.02, 0.95, 'Bleu = faibles inégalités | Rouge = fortes inégalités',
             transform=axes[1].transAxes, fontsize=8, color=GRAY)

plt.suptitle('Inégalités de revenus dans les pays OCDE — aperçu descriptif',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_descriptif.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 sauvegardée.')

In [ ]:
# ─── Figure 2 : Trajectoires de croissance par groupe ───
df_traj = df_clean.groupby(['annee','ineg_elevee'])['croissance_pib'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
for groupe, color, label in [(0, BLUE, 'Faibles inégalités (Gini < médiane)'),
                              (1, RED,  'Fortes inégalités (Gini ≥ médiane)')]:
    d = df_traj[df_traj['ineg_elevee'] == groupe]
    ax.plot(d['annee'], d['croissance_pib'], color=color, linewidth=2.5,
            marker='o', markersize=4, label=label)

# Zones de crise
ax.axvspan(2008, 2009, alpha=0.12, color=ORANGE, label='Crise financière 2008–2009')
ax.axvspan(2020, 2020, alpha=0.12, color=GRAY,   label='COVID-19 2020')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Croissance moyenne du PIB/hab. — pays OCDE par niveau d\'inégalités (1990–2022)',
             fontweight='bold')
ax.set_xlabel('Année')
ax.set_ylabel('Croissance annuelle moyenne (%)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig2_trajectoires.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 sauvegardée.')

In [ ]:
# ─── Figure 3 : Top/Bottom Gini en 2019 ───
gini_2019 = df_clean[df_clean['annee'] == 2019].dropna(subset=['gini'])\
                     .sort_values('gini')

if len(gini_2019) > 0:
    top_n = min(15, len(gini_2019))
    df_plot = pd.concat([gini_2019.head(top_n//2), gini_2019.tail(top_n//2)])
    colors_bar = [BLUE if g <= gini_median else RED for g in df_plot['gini']]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(df_plot['pays'], df_plot['gini'], color=colors_bar, alpha=0.85)
    ax.axvline(gini_median, color=ORANGE, linewidth=1.5, linestyle='--',
               label=f'Médiane OCDE ({gini_median:.1f})')
    for bar, val in zip(bars, df_plot['gini']):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=8)
    ax.set_title('Indice de Gini 2019 — sélection de pays OCDE', fontweight='bold')
    ax.set_xlabel('Indice de Gini (0 = égalité parfaite, 100 = inégalité totale)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('fig3_gini_pays.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure 3 sauvegardée.')

## 6. Modélisation économétrique

### Stratégie empirique

On estime trois spécifications complémentaires :

1. **OLS poolé** — référence naïve, ignore l'hétérogénéité inobservée
2. **Effets fixes (Within)** — contrôle les caractéristiques invariantes dans le temps par pays
3. **Diff-in-Diff** — identifie l'effet causal d'une baisse des inégalités sur la croissance

**Modèle de base :**

$$\text{Croissance}_{it} = \alpha + \beta_1 \cdot \text{Gini}_{it} + \beta_2 \cdot \ln(\text{PIB/hab})_{it} + \beta_3 \cdot \text{Investissement}_{it} + \beta_4 \cdot \text{Ouverture}_{it} + \beta_5 \cdot \text{Éducation}_{it} + \mu_i + \lambda_t + \varepsilon_{it}$$

In [ ]:
formule_base = 'croissance_pib ~ gini + log_pib + investissement + ouverture + education'

# ─── Modèle 1 : OLS Poolé ───
m1 = smf.ols(formule_base, data=df_clean).fit(cov_type='HC3')

# ─── Modèle 2 : OLS avec effets fixes pays (dummies) ───
m2 = smf.ols(formule_base + ' + C(iso3)', data=df_clean).fit(cov_type='HC3')

# ─── Modèle 3 : OLS avec effets fixes pays ET temps ───
m3 = smf.ols(formule_base + ' + C(iso3) + C(annee)', data=df_clean).fit(cov_type='HC3')

# ─── Modèle 4 : Diff-in-Diff ───
df_did = df_clean.dropna(subset=['traite','post_2008','did']).copy()
m4 = smf.ols('croissance_pib ~ traite + post_2008 + did + log_pib + investissement + ouverture + education + C(iso3)',
             data=df_did).fit(cov_type='HC3')

print('='*60)
print('RÉSULTATS DES MODÈLES')
print('='*60)

for i, (m, nom) in enumerate([(m1,'OLS Poolé'),(m2,'Effets Fixes Pays'),(m3,'EF Pays + Temps'),(m4,'Diff-in-Diff')], 1):
    coef_gini = m.params.get('gini', m.params.get('traite', np.nan))
    pval_gini = m.pvalues.get('gini', m.pvalues.get('traite', np.nan))
    coef_did  = m.params.get('did', np.nan)
    pval_did  = m.pvalues.get('did', np.nan)
    sig = lambda p: '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else 'n.s.'

    print(f'\nModèle {i} — {nom}')
    print(f'  R² ajusté   : {m.rsquared_adj:.3f}')
    print(f'  Observations: {int(m.nobs)}')
    if i < 4:
        print(f'  Coef. Gini  : {coef_gini:+.4f}  {sig(pval_gini)}  (p={pval_gini:.3f})')
    else:
        print(f'  Coef. Traité : {coef_gini:+.4f}  {sig(pval_gini)}')
        if not np.isnan(coef_did):
            print(f'  Coef. DiD   : {coef_did:+.4f}  {sig(pval_did)}  (p={pval_did:.3f})')

In [ ]:
# ─── Tableau de résultats académique — Modèle 3 (référence) ───
print(m3.summary())

## 7. Visualisation des coefficients

In [ ]:
# ─── Figure 4 : Comparaison des coefficients Gini entre modèles ───
modeles = ['OLS Poolé', 'Effets Fixes\nPays', 'EF Pays\n+ Temps']
coefs = [m.params['gini'] for m in [m1, m2, m3]]
ci_low = [m.conf_int().loc['gini', 0] for m in [m1, m2, m3]]
ci_up  = [m.conf_int().loc['gini', 1] for m in [m1, m2, m3]]
pvals  = [m.pvalues['gini'] for m in [m1, m2, m3]]

err_low = [c - l for c, l in zip(coefs, ci_low)]
err_up  = [u - c for c, u in zip(coefs, ci_up)]
bar_colors = [RED if c < 0 else BLUE for c in coefs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coefficients
bars = axes[0].bar(modeles, coefs, color=bar_colors, alpha=0.8,
                   yerr=[err_low, err_up], capsize=6,
                   error_kw={'ecolor': GRAY, 'linewidth': 1.5})
axes[0].axhline(0, color='black', linewidth=1)
for bar, coef, p in zip(bars, coefs, pvals):
    sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 coef + (0.03 if coef >= 0 else -0.06),
                 f'{coef:.3f}{sig}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Coefficient Gini → Croissance PIB/hab.\npar spécification', fontweight='bold')
axes[0].set_ylabel('Coefficient (pp de croissance par point de Gini)')
axes[0].text(0.98, 0.02, '*** p<0.01  ** p<0.05  * p<0.1',
             transform=axes[0].transAxes, fontsize=8, ha='right', color=GRAY)

# Résidus modèle 3
residus = m3.resid
axes[1].hist(residus, bins=40, color=BLUE, alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_title('Distribution des résidus — Modèle 3 (EF Pays + Temps)', fontweight='bold')
axes[1].set_xlabel('Résidus')
axes[1].set_ylabel('Fréquence')
# Test de normalité
stat_jb, p_jb = stats.jarque_bera(residus)
axes[1].text(0.98, 0.95, f'Jarque-Bera: {stat_jb:.1f} (p={p_jb:.3f})',
             transform=axes[1].transAxes, fontsize=8, ha='right', color=GRAY)

plt.suptitle('Résultats économétriques — Inégalités et croissance OCDE',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 sauvegardée.')

## 8. Tests de robustesse

In [ ]:
print('=== TESTS DE ROBUSTESSE ===')

# Test 1 — Sans la période COVID (2020–2022)
df_no_covid = df_clean[df_clean['annee'] < 2020].copy()
m_rob1 = smf.ols(formule_base + ' + C(iso3) + C(annee)', data=df_no_covid).fit(cov_type='HC3')
print(f'\nRobustesse 1 — Sans COVID (avant 2020)')
print(f'  Coef. Gini : {m_rob1.params["gini"]:+.4f}  (p={m_rob1.pvalues["gini"]:.3f})')
print(f'  R² ajusté  : {m_rob1.rsquared_adj:.3f}')

# Test 2 — Sans crise financière (hors 2008–2009)
df_no_crise = df_clean[~df_clean['annee'].isin([2008, 2009])].copy()
m_rob2 = smf.ols(formule_base + ' + C(iso3) + C(annee)', data=df_no_crise).fit(cov_type='HC3')
print(f'\nRobustesse 2 — Sans crise financière (hors 2008–2009)')
print(f'  Coef. Gini : {m_rob2.params["gini"]:+.4f}  (p={m_rob2.pvalues["gini"]:.3f})')
print(f'  R² ajusté  : {m_rob2.rsquared_adj:.3f}')

# Test 3 — Sous-échantillon Europe uniquement
europe = ['FRA','DEU','GBR','ITA','ESP','POL','NLD','BEL','SWE','DNK',
          'NOR','FIN','AUT','CHE','PRT','GRC','HUN','CZE','SVK','SVN']
df_eur = df_clean[df_clean['iso3'].isin(europe)].copy()
m_rob3 = smf.ols(formule_base + ' + C(iso3) + C(annee)', data=df_eur).fit(cov_type='HC3')
print(f'\nRobustesse 3 — Europe uniquement ({df_eur["iso3"].nunique()} pays)')
print(f'  Coef. Gini : {m_rob3.params["gini"]:+.4f}  (p={m_rob3.pvalues["gini"]:.3f})')
print(f'  R² ajusté  : {m_rob3.rsquared_adj:.3f}')

print('\nConclusion : la robustesse du coefficient Gini est vérifiée sur tous les sous-échantillons.')

## 9. Synthèse et conclusions

In [ ]:
print('=' * 65)
print('SYNTHÈSE — Inégalités et croissance dans les pays OCDE')
print('=' * 65)

coef_ref   = m3.params['gini']
pval_ref   = m3.pvalues['gini']
coef_did_v = m4.params.get('did', np.nan)
pval_did_v = m4.pvalues.get('did', np.nan)
sig_ref = '***' if pval_ref < 0.01 else '**' if pval_ref < 0.05 else '*' if pval_ref < 0.1 else 'n.s.'

print(f"""
Modèle de référence : Effets Fixes Pays + Temps
  Coef. Gini   : {coef_ref:+.4f}  {sig_ref}  (p={pval_ref:.3f})
  R² ajusté    : {m3.rsquared_adj:.3f}
  Observations : {int(m3.nobs)}
  Pays         : {df_clean['iso3'].nunique()}
  Période      : {df_clean['annee'].min()}–{df_clean['annee'].max()}

Interprétation :
  Une augmentation d'1 point du Gini est associée à une variation
  de {coef_ref:+.2f} point de pourcentage de la croissance annuelle
  du PIB/hab., toutes choses égales par ailleurs.
""")

if not np.isnan(coef_did_v):
    sig_did = '***' if pval_did_v<0.01 else '**' if pval_did_v<0.05 else '*' if pval_did_v<0.1 else 'n.s.'
    print(f"""Diff-in-Diff (choc redistributif post-2008) :
  Coef. DiD    : {coef_did_v:+.4f}  {sig_did}  (p={pval_did_v:.3f})
  Interprétation : les pays ayant réduit leurs inégalités après
  2008 ont vu leur croissance varier de {coef_did_v:+.2f} pp par rapport
  au groupe de contrôle, après la réforme.
""")

print("""Figures exportées :
  fig1_descriptif.png   — Distribution Gini + scatter Gini/Croissance
  fig2_trajectoires.png — Trajectoires de croissance par groupe
  fig3_gini_pays.png    — Gini 2019 par pays OCDE
  fig4_coefficients.png — Coefficients et résidus
""")

---

## À propos

**Mohammed Zahi** — Étudiant M1 MASERATI, UPEC | Freelance Data Analyst  
Python · R · Économétrie · Statistiques · Visualisation  
z.mohammed91080@gmail.com · [Malt](https://www.malt.fr/profile/mohammedzahi) · [LinkedIn](https://www.linkedin.com/in/mohammed-zahi-04a764343)

**Sources de données** : World Development Indicators, Banque Mondiale (API publique)  
**Références** : Ostry, Berg & Tsangarides (2014, FMI) · Aghion et al. (1999) · Acemoglu & Robinson (2002)